<a href="https://colab.research.google.com/github/Inventor-creator/Sentiment-Analysis-ML-proj/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Preproccessing

first project, so have searched google on how to convert the .txt file into a dataset


In [121]:

import re

def convertToDataset():
    labels = []
    reviews = []
    i = 10000
    with open('test.ft.txt', 'r', encoding= "utf-8") as f:

        for currLine in f :
            if i > 0:
                match = re.match(r"^(__label__\d+)\s+(.*)", currLine)

                if match:
                    label = 0 if match.group(1) == '__label__1' else 1
                    review = match.group(2)


                    review = review.replace('\\' , "")



                    # print("Label:", label)
                    labels.append(label)
                    # print("Review:", review)
                    reviews.append(review)
                    i-=1


    return labels , reviews

labels , reviews = convertToDataset()


In [122]:
print(labels.count(1))
print(labels.count(0))


5125
4875


In [123]:
from sklearn.model_selection import train_test_split

xTrain, xTest, yTrain, yTest = train_test_split(reviews , labels , random_state=42)
xTrain2, xTest2, yTrain2, yTest2 = train_test_split(reviews , labels , random_state=42)
xTrainCNN, xTestCNN, yTrainCNN, yTestCNN = train_test_split(reviews , labels , random_state=42)



Tried Normal vectorizer without df args -> 56 percent score
with max_df now 75 percent


In [124]:
from sklearn.model_selection import GridSearchCV

In [125]:
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

In [126]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_df=0.8 , min_df=0.05)

xTrain = vectorizer.fit_transform(xTrain)
xTest = vectorizer.transform(xTest)

xTrain = xTrain.toarray()
xTest = xTest.toarray()

In [127]:
from sklearn.naive_bayes import GaussianNB

gaussianModel = GaussianNB()

gaussianModel.fit(xTrain, yTrain)


,priors,None
,var_smoothing,1e-09


0.8 didnt move on either side
and minDf moves lower
added extra 0.0005 to test, but seems that 0.005 is best for vectorizer


In [128]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

gridSearchVectorizer = TfidfVectorizer()

modelEncodePipe = Pipeline(steps=[
    ('vectorizer', gridSearchVectorizer),
    ('convert', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('model' , GaussianNB())
])

param_grid = {
    'vectorizer__max_df': [0.8, 0.5, 1.0],
    'vectorizer__min_df': [0.1, 0.05 , 0.005,0.0005],
}

gridSearchModel = GridSearchCV(estimator=modelEncodePipe ,param_grid=param_grid , cv=5)
gridSearchModel.fit(xTrain2, yTrain2)



,estimator,Pipeline(step...aussianNB())])
,param_grid,"{'vectorizer__max_df': [0.8, 0.5, ...], 'vectorizer__min_df': [0.1, 0.05, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [129]:
gridSearchModel.best_params_

{'vectorizer__max_df': 0.8, 'vectorizer__min_df': 0.005}

In [130]:
yPred = gridSearchModel.predict(xTest2)

#actual true true or not
print(recall_score(yTest2, yPred ))
print(accuracy_score(yTest2, yPred ))
print(roc_auc_score(yTest2, yPred ))
print(confusion_matrix(yTest2 , yPred) / len(yTest2))

0.8034188034188035
0.8
0.7998957166310835
[[0.3864 0.0988]
 [0.1012 0.4136]]


In [131]:
yPred = gaussianModel.predict(xTest)

#actual true true or not
print(recall_score(yTest, yPred ))
print(accuracy_score(yTest, yPred ))
print(roc_auc_score(yTest, yPred ))
print(confusion_matrix(yTest , yPred) / len(yTest))

0.7156177156177156
0.7088
0.708592040001768
[[0.3404 0.1448]
 [0.1464 0.3684]]


Trying a new model

In [132]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")


Using device: mps


In [133]:
xTrain.shape

(7500, 166)

In [135]:
import numpy as np

yTrain = np.array(yTrain)
yTest = np.array(yTest)

In [136]:
xTrainCNN = netVectorizer.fit_transform(xTrainCNN).toarray()
xTestCNN = netVectorizer.transform(xTestCNN).toarray()


In [144]:
from torch import nn
from torch.optim import Adam



netVectorizer = TfidfVectorizer()


class Net1(nn.Module):

    def __init__(self, input_dim=166, output_dim=1):
        super(Net1, self).__init__()

        # Layer 1: Input to Hidden 1
        self.fc1 = nn.Linear(input_dim, 115)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.2)

        # Layer 2: Hidden 1 to Hidden 2
        self.fc2 = nn.Linear(115, 60)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.3)

        # Layer 3: Hidden 2 to Output
        # (Note: PyTorch's BCEWithLogitsLoss expects raw logits, so no Sigmoid here)
        self.out = nn.Linear(60, output_dim)


    def forward(self, x):
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        x = self.out(x)
        return x

netModel = Net1(input_dim=xTrainCNN.shape[1])
optimizer = Adam(model.parameters(), lr=0.1)
loss = nn.BCEWithLogitsLoss()



In [137]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(torch.tensor(xTrainCNN  ), torch.tensor(yTrain.reshape(yTrain.shape[0] , -1)))
dataloader = DataLoader(dataset, batch_size=32 ,shuffle=True)




In [146]:
netModel.to(device)

Net1(
  (fc1): Linear(in_features=29137, out_features=115, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.2, inplace=False)
  (fc2): Linear(in_features=115, out_features=60, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (out): Linear(in_features=60, out_features=1, bias=True)
)

In [149]:
epochs = 100

epochNum = []
lossatNum = []


for i in range(epochs):
    for inputs,targets in dataloader:

        inputs = inputs.to(device, dtype=torch.float32, non_blocking=True)
        targets = targets.to(device, dtype=torch.float32, non_blocking=True)

        optimizer.zero_grad()
        outputs = netModel.forward(inputs.to(torch.float32))
        # print(outputs)
        loss_value = loss(outputs, targets.to(torch.float32))
        loss_value.backward()
        optimizer.step()
        # scheduler.step(loss_value)

    epochNum.append(i+1)
    lossatNum.append(loss_value.item())
    print(f"Epoch {i+1}/{epochs}, Loss: {loss_value.item()}")


tensor([[1.6426e+20],
        [4.8245e+07],
        [1.4349e-42],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00]], device='mps:0')
tensor([[1.6404e+20],
        [4.8245e+07],
        [1.4349e-42],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000


KeyboardInterrupt



CNN approach here



In [ ]:
import nltk
from nltk.tokenize import word_tokenize


In [ ]:
#tokenize shti
import re
tokenizedReviews = []

for i in xTrainCNN:
    i = i.lower()
    i = re.sub(r'[^a-z0-9\s]', '', i)
    tokenizedReviewsTemp = word_tokenize(i)

    tokenizedReviews.append(tokenizedReviewsTemp)


counter = 0
tokenDict = {}
tokenDict["<PAD>"] = 0
tokenDict["<UNK>"] = 1

from torch.utils.data import Dataset, DataLoader
import nltk

class TextCNNDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length=20):
        self.labels = labels  # Your numerical labels list
        self.max_length = max_length
        self.data = []

        # Tokenize and index text
        for text in texts:
            tokens = nltk.word_tokenize(text.lower())
            indexed = [vocab.get(token, 1) for token in tokens] # 1 is <UNK>

            # Pad or truncate to fixed length
            if len(indexed) < max_length:
                padded = indexed + [0] * (max_length - len(indexed)) # 0 is <PAD>
            else:
                padded = indexed[:max_length]

            self.data.append(padded)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # PyTorch CNNs expect LongTensors for text IDs and integer labels
        return {
            'text_tensors': torch.tensor(self.data[idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }






In [ ]:
tokenizedReviews[0]

In [ ]:
from torch import nn



class CNNCLASSIFIER(nn.Module):
    def __init__(self ,vocab_size,embedding_dim ,inFeatures, hiddenConvFeatures1 , hidden):
        super(CNNCLASSIFIER, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.inputLayer = nn.Linear(in_features= inFeatures , out_features=hiddenConvFeatures1)
        self.fc1 = nn.Conv1d(in_channels=hiddenConvFeatures1 )

